## Task 1: Regression network training and testing with PyTorch Lightning

#### Goal: Implement a regression MLP using PyTorch Lightning on a toy dataset.

1. Install the required libraries: `pip install pytorch-lightning` and `pip install ray[tune]`.
2. Implement a `Dataset` subclass (`MyDataset`) that stores input features and targets and returns individual samples via `__getitem__`.
3. Implement a `LightningDataModule` (`MyDataModule`) that generates a regression toy dataset using `make_regression`, converts it to `float32` tensors, splits it into train/val/test subsets, and exposes them via `train_dataloader`, `val_dataloader`, and `test_dataloader`.
4. Implement a `LightningModule` (`LitModel`) with a 3-layer MLP. Use `self.save_hyperparameters()` in `__init__`. Implement `training_step`, `validation_step`, and `test_step` — each should compute MSE loss; validation and test steps should also compute and log R² score. Implement `configure_optimizers` returning an Adam optimizer using `self.hparams.lr`.
5. Instantiate a `Trainer` with `max_epochs` and appropriate `accelerator` settings and run `trainer.fit()` followed by `trainer.test()`.
6. Add an `EarlyStopping` callback (monitoring `val_loss` with patience 5) and a `ModelCheckpoint` callback (saving the top 3 checkpoints by `val_loss` plus the last one). Re-run training with both callbacks active.
7. Define a `train_model(config)` function for Ray Tune that instantiates `LitModel` with sampled `hidden_size` and `lr` values and trains it using a `TuneReportCallback`. Run the hyperparameter search using `tune.run()` with `tune.choice` for `hidden_size` and `tune.loguniform` for `lr` over 10 samples, then print the best config.

**Assignment:** Implement and run hyperparameter optimization for a regression network on the toy dataset using PyTorch Lightning.


In [3]:
%load_ext autoreload
%autoreload 2

In [1]:
from utils import *
import torch

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [3]:
import pytorch_lightning as pl

n_features = 12

datamodule = MyDataModule(batch_size = 32, n_samples = 8000, n_features = n_features)
model = LitModel(hidden_layers = [64, 256, 64], input_dim = n_features, output_dim = 1)

trainer = pl.Trainer(max_epochs=10, accelerator=device, devices=1)
trainer.fit(model, datamodule=datamodule)
trainer.test(model, datamodule=datamodule)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name          | Type       | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | input_layer   | Linear     | 832    | train | 0    
1 | hidden_layers | ModuleList | 33.1 K | train | 0    
2 | output_layer  | Linear     | 65     | train | 0    
3 | activation    | ReLU       | 0      | train | 0    
4 | criterion     | MSELoss    | 0      | train | 0    
-------------------------------------------------------------
34.0 K    Trainable param

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/dominikmika/PycharmProjects/Deep-learning-lab/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss           21.174951553344727
         test_r2            0.9993381500244141
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 21.174951553344727, 'test_r2': 0.9993381500244141}]

In [4]:
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

early_stop_callback = EarlyStopping(monitor='val_loss', patience=5, verbose=True, mode='min')

checkpoint_callback = ModelCheckpoint(dirpath='./', filename='{epoch:02d}-{val_loss:.2f}', monitor='val_loss', save_top_k=3, mode='min', save_last=True)

trainer = pl.Trainer(callbacks=[early_stop_callback, checkpoint_callback], max_epochs=100, accelerator=device, devices=1)

trainer.fit(model, datamodule=datamodule)
trainer.test(model, datamodule=datamodule)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/dominikmika/PycharmProjects/Deep-learning-lab/.venv/lib/python3.13/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/dominikmika/PycharmProjects/Deep-learning-lab/Lab11 exists and is not empty.

  | Name          | Type       | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | input_layer   | Linear     | 832    | train | 0    
1 | hidden_layers | ModuleList | 33.1 K | train | 0    
2 | output_layer  | Linear     | 65     | train | 0    
3 | activation    | ReLU       | 0      | train | 0    
4 | criterion     | MSELoss    | 0      | train | 0    
----------------------------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_loss improved. New best score: 19.558


...

Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_loss did not improve in the last 5 records. Best score: 2.050. Signaling Trainer to stop.


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss            3.224606990814209
         test_r2            0.9999076724052429
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 3.224606990814209, 'test_r2': 0.9999076724052429}]

In [ ]:
import ray
from ray import tune

ray.init(object_store_memory=2*1024*1024*1024)

analysis = tune.run(train_model, num_samples=10,
                    config={'hidden_layers': tune.choice([[64, 256, 64], [32, 64, 128]]),
                                         'input_dim': n_features, 'output_dim': 1,
                                         'max_epochs': 10, 'lr': tune.loguniform(1e-4, 1e-2), 'n_samples': 8000, 'batch_size': 32}
                    )

print(analysis.get_best_config(metric='loss', mode='min'))

2026-06-23 02:49:02,689	INFO worker.py:2012 -- Started a local Ray instance.
/Users/dominikmika/PycharmProjects/Deep-learning-lab/.venv/lib/python3.13/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
2026-06-23 02:49:03,577	INFO tune.py:615 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


(train_model pid=1592) /Users/dominikmika/PycharmProjects/Deep-learning-lab/.venv/lib/python3.13/site-packages/ray/tune/integration/pytorch_lightning.py:198: `ray.tune.integration.pytorch_lightning.TuneReportCallback` is deprecated. Use `ray.tune.integration.pytorch_lightning.TuneReportCheckpointCallback` instead.
(train_model pid=1592) GPU available: True (mps), used: True
(train_model pid=1592) TPU available: False, using: 0 TPU cores
(train_model pid=1592) 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
(train_model pid=1592) 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
(train_model pid=1592) 
(train_model pid=1592)   | Name          | Type       

Sanity Checking: |          | 0/? [00:00<?, ?it/s]
